# AIC Collection — SQL Query Deliverable
**Author:** Jillian Robertson · Spring 2026

This notebook is the **separate SQL deliverable** that accompanies the polished analysis
in `aic_collection_final_robertson.ipynb`.It loads the cached AIC
catalogue, does the minimal cleaning needed to recreate the `region`, `era`, and
`has_exhibition_history` fields, loads everything into an in-memory **SQLite** database,
and then runs the required SQL queries.

## Setup & ETL
Load the catalogue, rebuild the derived fields, and stage two tables in SQLite.

In [1]:
import pandas as pd, numpy as np, sqlite3
from pathlib import Path

# ---- Load the cached full catalogue (same file the Python notebook uses) ----
CSV_PATH = Path("aic_output/aic_artworks_full.csv")
art = pd.read_csv(CSV_PATH, low_memory=False)

# Normalise empty lists to NaN, then derive the same analytic fields used in the
# main notebook so the SQL numbers line up exactly with the Python analysis.
art = art.map(lambda x: np.nan if isinstance(x, list) and len(x) == 0 else x)

# year_start: the AIC API ships numeric date_start; fall back gracefully if absent.
art["year_start"] = pd.to_numeric(art.get("date_start"), errors="coerce")
art["year_end"]   = pd.to_numeric(art.get("date_end"), errors="coerce")

# region (identical mapping to the main notebook)
# Outside dataset: country -> world-region mapping (UN geoscheme + art-historical regions)
country_to_region = {
    # ============ NORTH AMERICA ============
    "United States": "North America", "USA": "North America", "U.S.A.": "North America",
    "U.S.": "North America", "America": "North America", "American": "North America",
    "Native American": "North America",
    "Canada": "North America", "Canadian": "North America",
    "Mexico": "North America", "Mexican": "North America",
    # All 50 US states (so "Illinois", "California" etc. map directly)
    "Alabama": "North America", "Alaska": "North America", "Arizona": "North America",
    "Arkansas": "North America", "California": "North America", "Colorado": "North America",
    "Connecticut": "North America", "Delaware": "North America", "Florida": "North America",
    "Georgia": "North America", "Idaho": "North America", "Illinois": "North America",
    "Indiana": "North America", "Iowa": "North America", "Kansas": "North America",
    "Kentucky": "North America", "Louisiana": "North America", "Maine": "North America",
    "Maryland": "North America", "Massachusetts": "North America", "Michigan": "North America",
    "Minnesota": "North America", "Mississippi": "North America", "Missouri": "North America",
    "Montana": "North America", "Nebraska": "North America", "Nevada": "North America",
    "New Hampshire": "North America", "New Jersey": "North America", "New Mexico": "North America",
    "New York": "North America", "North Carolina": "North America", "North Dakota": "North America",
    "Ohio": "North America", "Oklahoma": "North America", "Oregon": "North America",
    "Pennsylvania": "North America", "Rhode Island": "North America", "South Carolina": "North America",
    "South Dakota": "North America", "Tennessee": "North America", "Texas": "North America",
    "Utah": "North America", "Vermont": "North America", "Virginia": "North America",
    "Washington": "North America", "West Virginia": "North America", "Wisconsin": "North America",
    "Wyoming": "North America",
    # Major US art-center cities
    "Chicago": "North America", "Boston": "North America", "Philadelphia": "North America",
    "Los Angeles": "North America", "San Francisco": "North America", "Detroit": "North America",
    "Cleveland": "North America", "Cincinnati": "North America", "Pittsburgh": "North America",
    "Baltimore": "North America", "Atlanta": "North America", "New Orleans": "North America",
    "Saint Louis": "North America", "St. Louis": "North America", "Milwaukee": "North America",
    "Seattle": "North America", "Portland": "North America", "Minneapolis": "North America",
    "Denver": "North America", "Santa Fe": "North America", "Taos": "North America",
    "Hartford": "North America", "Providence": "North America", "Buffalo": "North America",
    "Brooklyn": "North America", "Manhattan": "North America", "Indianapolis": "North America",
    "Houston": "North America", "Dallas": "North America", "Miami": "North America",
    "Phoenix": "North America",
    # Canadian cities
    "Toronto": "North America", "Montreal": "North America", "Vancouver": "North America",
    "Quebec": "North America", "Ottawa": "North America",

    # ============ LATIN AMERICA & CARIBBEAN ============
    "Latin America": "Latin America & Caribbean", "South America": "Latin America & Caribbean",
    "Central America": "Latin America & Caribbean", "Caribbean": "Latin America & Caribbean",
    "Pre-Columbian": "Latin America & Caribbean",
    "Peru": "Latin America & Caribbean", "Peruvian": "Latin America & Caribbean",
    "Brazil": "Latin America & Caribbean", "Brazilian": "Latin America & Caribbean",
    "Argentina": "Latin America & Caribbean", "Argentinian": "Latin America & Caribbean",
    "Argentinean": "Latin America & Caribbean",
    "Chile": "Latin America & Caribbean", "Chilean": "Latin America & Caribbean",
    "Colombia": "Latin America & Caribbean", "Colombian": "Latin America & Caribbean",
    "Cuba": "Latin America & Caribbean", "Cuban": "Latin America & Caribbean",
    "Guatemala": "Latin America & Caribbean", "Guatemalan": "Latin America & Caribbean",
    "Bolivia": "Latin America & Caribbean", "Bolivian": "Latin America & Caribbean",
    "Ecuador": "Latin America & Caribbean", "Ecuadorian": "Latin America & Caribbean",
    "Venezuela": "Latin America & Caribbean", "Venezuelan": "Latin America & Caribbean",
    "Uruguay": "Latin America & Caribbean", "Paraguay": "Latin America & Caribbean",
    "Costa Rica": "Latin America & Caribbean", "Panama": "Latin America & Caribbean",
    "Honduras": "Latin America & Caribbean", "Nicaragua": "Latin America & Caribbean",
    "El Salvador": "Latin America & Caribbean",
    "Haiti": "Latin America & Caribbean", "Haitian": "Latin America & Caribbean",
    "Jamaica": "Latin America & Caribbean", "Jamaican": "Latin America & Caribbean",
    "Puerto Rico": "Latin America & Caribbean", "Dominican Republic": "Latin America & Caribbean",
    "Trinidad": "Latin America & Caribbean", "Bahamas": "Latin America & Caribbean",
    "Barbados": "Latin America & Caribbean",
    # Pre-Columbian / indigenous cultures
    "Aztec": "Latin America & Caribbean", "Mayan": "Latin America & Caribbean",
    "Maya": "Latin America & Caribbean", "Olmec": "Latin America & Caribbean",
    "Inca": "Latin America & Caribbean", "Incan": "Latin America & Caribbean",
    "Mesoamerica": "Latin America & Caribbean", "Mesoamerican": "Latin America & Caribbean",
    "Teotihuacan": "Latin America & Caribbean", "Moche": "Latin America & Caribbean",
    "Nazca": "Latin America & Caribbean", "Chimu": "Latin America & Caribbean",
    "Tiwanaku": "Latin America & Caribbean", "Yucatan": "Latin America & Caribbean",
    "Oaxaca": "Latin America & Caribbean", "Veracruz": "Latin America & Caribbean",
    # Major cities
    "Mexico City": "Latin America & Caribbean", "Lima": "Latin America & Caribbean",
    "Cusco": "Latin America & Caribbean", "Cuzco": "Latin America & Caribbean",
    "Buenos Aires": "Latin America & Caribbean", "Rio de Janeiro": "Latin America & Caribbean",
    "São Paulo": "Latin America & Caribbean", "Sao Paulo": "Latin America & Caribbean",
    "Havana": "Latin America & Caribbean", "Bogotá": "Latin America & Caribbean",
    "Bogota": "Latin America & Caribbean", "Santiago": "Latin America & Caribbean",

    # ============ EUROPE ============
    "France": "Europe", "French": "Europe",
    "Italy": "Europe", "Italian": "Europe",
    "Germany": "Europe", "German": "Europe",
    "England": "Europe", "English": "Europe",
    "United Kingdom": "Europe", "Britain": "Europe", "British": "Europe", "Great Britain": "Europe",
    "Scotland": "Europe", "Scottish": "Europe",
    "Wales": "Europe", "Welsh": "Europe",
    "Ireland": "Europe", "Irish": "Europe",
    "Spain": "Europe", "Spanish": "Europe",
    "Portugal": "Europe", "Portuguese": "Europe",
    "Netherlands": "Europe", "Dutch": "Europe", "Holland": "Europe",
    "Belgium": "Europe", "Belgian": "Europe", "Flanders": "Europe", "Flemish": "Europe",
    "Switzerland": "Europe", "Swiss": "Europe",
    "Austria": "Europe", "Austrian": "Europe",
    "Greece": "Europe", "Greek": "Europe", "Hellenistic": "Europe",
    "Russia": "Europe", "Russian": "Europe",
    "Poland": "Europe", "Polish": "Europe",
    "Sweden": "Europe", "Swedish": "Europe",
    "Norway": "Europe", "Norwegian": "Europe",
    "Denmark": "Europe", "Danish": "Europe",
    "Finland": "Europe", "Finnish": "Europe",
    "Iceland": "Europe", "Icelandic": "Europe",
    "Hungary": "Europe", "Hungarian": "Europe",
    "Czech Republic": "Europe", "Czech": "Europe", "Czechoslovakia": "Europe",
    "Slovakia": "Europe", "Slovak": "Europe",
    "Romania": "Europe", "Romanian": "Europe",
    "Bulgaria": "Europe", "Bulgarian": "Europe",
    "Ukraine": "Europe", "Ukrainian": "Europe",
    "Serbia": "Europe", "Croatia": "Europe", "Croatian": "Europe",
    "Slovenia": "Europe", "Bosnia": "Europe", "Albania": "Europe",
    "Lithuania": "Europe", "Latvia": "Europe", "Estonia": "Europe",
    "Belarus": "Europe", "Moldova": "Europe", "Malta": "Europe",
    "Cyprus": "Europe", "Luxembourg": "Europe",
    # Pre-modern European entities & regions
    "Bohemia": "Europe", "Bohemian": "Europe", "Moravia": "Europe",
    "Prussia": "Europe", "Prussian": "Europe", "Bavaria": "Europe", "Bavarian": "Europe",
    "Saxony": "Europe", "Saxon": "Europe", "Westphalia": "Europe",
    "Lombardy": "Europe", "Lombard": "Europe", "Tuscany": "Europe", "Tuscan": "Europe",
    "Sicily": "Europe", "Sicilian": "Europe", "Sardinia": "Europe",
    "Piedmont": "Europe", "Umbria": "Europe", "Veneto": "Europe", "Liguria": "Europe",
    "Catalonia": "Europe", "Catalan": "Europe", "Andalusia": "Europe", "Andalusian": "Europe",
    "Castile": "Europe", "Castilian": "Europe", "Aragon": "Europe", "Aragonese": "Europe",
    "Navarre": "Europe", "Galicia": "Europe", "Basque": "Europe",
    "Brittany": "Europe", "Breton": "Europe", "Normandy": "Europe", "Norman": "Europe",
    "Provence": "Europe", "Burgundy": "Europe", "Burgundian": "Europe",
    "Etruria": "Europe", "Etruscan": "Europe",
    "Roman Empire": "Europe", "Roman": "Europe",
    "Byzantine": "Europe", "Byzantium": "Europe",
    "Holy Roman Empire": "Europe",
    "Yugoslavia": "Europe", "Soviet Union": "Europe", "USSR": "Europe",
    # Major cities
    "Paris": "Europe", "Lyon": "Europe", "Marseille": "Europe", "Bordeaux": "Europe",
    "Avignon": "Europe", "Nice": "Europe", "Strasbourg": "Europe", "Toulouse": "Europe",
    "Lille": "Europe",
    "Rome": "Europe", "Florence": "Europe", "Florentine": "Europe", "Venice": "Europe",
    "Venetian": "Europe", "Milan": "Europe", "Milanese": "Europe", "Naples": "Europe",
    "Neapolitan": "Europe", "Bologna": "Europe", "Siena": "Europe", "Sienese": "Europe",
    "Pisa": "Europe", "Padua": "Europe", "Genoa": "Europe", "Verona": "Europe", "Turin": "Europe",
    "Ravenna": "Europe", "Pompeii": "Europe", "Herculaneum": "Europe", "Mantua": "Europe",
    "Ferrara": "Europe", "Urbino": "Europe", "Parma": "Europe",
    "London": "Europe", "Edinburgh": "Europe", "Glasgow": "Europe", "Manchester": "Europe",
    "Liverpool": "Europe", "Oxford": "Europe", "Cambridge": "Europe", "Bath": "Europe",
    "Bristol": "Europe", "York": "Europe",
    "Berlin": "Europe", "Munich": "Europe", "Hamburg": "Europe", "Dresden": "Europe",
    "Cologne": "Europe", "Frankfurt": "Europe", "Düsseldorf": "Europe", "Dusseldorf": "Europe",
    "Stuttgart": "Europe", "Nuremberg": "Europe", "Nürnberg": "Europe", "Leipzig": "Europe",
    "Weimar": "Europe", "Augsburg": "Europe",
    "Amsterdam": "Europe", "Rotterdam": "Europe", "The Hague": "Europe", "Hague": "Europe",
    "Haarlem": "Europe", "Utrecht": "Europe", "Leiden": "Europe", "Delft": "Europe",
    "Madrid": "Europe", "Barcelona": "Europe", "Seville": "Europe", "Valencia": "Europe",
    "Toledo": "Europe", "Granada": "Europe", "Cordoba": "Europe", "Bilbao": "Europe",
    "Lisbon": "Europe", "Porto": "Europe",
    "Vienna": "Europe", "Salzburg": "Europe",
    "Brussels": "Europe", "Antwerp": "Europe", "Bruges": "Europe", "Ghent": "Europe",
    "Geneva": "Europe", "Zurich": "Europe", "Basel": "Europe", "Bern": "Europe",
    "Athens": "Europe", "Crete": "Europe", "Corinth": "Europe", "Mycenae": "Europe",
    "Sparta": "Europe",
    "Moscow": "Europe", "St. Petersburg": "Europe", "Saint Petersburg": "Europe",
    "Kyiv": "Europe", "Kiev": "Europe",
    "Prague": "Europe", "Warsaw": "Europe", "Krakow": "Europe", "Cracow": "Europe",
    "Budapest": "Europe",
    "Stockholm": "Europe", "Copenhagen": "Europe", "Oslo": "Europe", "Helsinki": "Europe",
    "Reykjavik": "Europe", "Dublin": "Europe", "Belfast": "Europe", "Cardiff": "Europe",

    # ============ EAST ASIA ============
    "Japan": "East Asia", "Japanese": "East Asia",
    "China": "East Asia", "Chinese": "East Asia",
    "Korea": "East Asia", "Korean": "East Asia",
    "South Korea": "East Asia", "North Korea": "East Asia",
    "Taiwan": "East Asia", "Taiwanese": "East Asia",
    "Mongolia": "East Asia", "Mongolian": "East Asia", "Mongol": "East Asia",
    "Hong Kong": "East Asia", "Tibet": "East Asia", "Tibetan": "East Asia",
    # Chinese dynasties
    "Tang Dynasty": "East Asia", "Tang": "East Asia",
    "Ming Dynasty": "East Asia", "Ming": "East Asia",
    "Qing Dynasty": "East Asia", "Qing": "East Asia",
    "Han Dynasty": "East Asia", "Han": "East Asia",
    "Song Dynasty": "East Asia", "Song": "East Asia",
    "Yuan Dynasty": "East Asia", "Yuan": "East Asia",
    "Zhou": "East Asia", "Shang": "East Asia",
    # Japanese periods
    "Edo": "East Asia", "Meiji": "East Asia", "Heian": "East Asia",
    "Kamakura": "East Asia", "Muromachi": "East Asia", "Momoyama": "East Asia",
    "Nara": "East Asia", "Taisho": "East Asia", "Showa": "East Asia",
    # Cities
    "Tokyo": "East Asia", "Kyoto": "East Asia", "Osaka": "East Asia",
    "Beijing": "East Asia", "Peking": "East Asia", "Shanghai": "East Asia",
    "Canton": "East Asia", "Guangzhou": "East Asia", "Nanjing": "East Asia",
    "Xian": "East Asia", "Xi'an": "East Asia",
    "Seoul": "East Asia", "Joseon": "East Asia", "Goryeo": "East Asia",

    # ============ SOUTH & SE ASIA ============
    "India": "South & SE Asia", "Indian": "South & SE Asia",
    "Pakistan": "South & SE Asia", "Pakistani": "South & SE Asia",
    "Bangladesh": "South & SE Asia", "Sri Lanka": "South & SE Asia", "Ceylon": "South & SE Asia",
    "Nepal": "South & SE Asia", "Nepalese": "South & SE Asia", "Bhutan": "South & SE Asia",
    "Thailand": "South & SE Asia", "Thai": "South & SE Asia", "Siam": "South & SE Asia",
    "Vietnam": "South & SE Asia", "Vietnamese": "South & SE Asia",
    "Indonesia": "South & SE Asia", "Indonesian": "South & SE Asia",
    "Java": "South & SE Asia", "Javanese": "South & SE Asia", "Bali": "South & SE Asia",
    "Sumatra": "South & SE Asia",
    "Cambodia": "South & SE Asia", "Cambodian": "South & SE Asia", "Khmer": "South & SE Asia",
    "Philippines": "South & SE Asia", "Filipino": "South & SE Asia",
    "Myanmar": "South & SE Asia", "Burma": "South & SE Asia", "Burmese": "South & SE Asia",
    "Laos": "South & SE Asia", "Malaysia": "South & SE Asia", "Malaysian": "South & SE Asia",
    "Singapore": "South & SE Asia",
    "Afghanistan": "South & SE Asia", "Afghan": "South & SE Asia",
    # Indian regions/dynasties
    "Mughal": "South & SE Asia", "Rajput": "South & SE Asia", "Pala": "South & SE Asia",
    "Gandhara": "South & SE Asia", "Bengal": "South & SE Asia", "Bengali": "South & SE Asia",
    "Punjab": "South & SE Asia", "Punjabi": "South & SE Asia", "Rajasthan": "South & SE Asia",
    "Kashmir": "South & SE Asia", "Gujarat": "South & SE Asia",
    "Tamil Nadu": "South & SE Asia", "Tamil": "South & SE Asia", "Kerala": "South & SE Asia",
    "Andhra Pradesh": "South & SE Asia", "Karnataka": "South & SE Asia",
    "Madhya Pradesh": "South & SE Asia", "Uttar Pradesh": "South & SE Asia",
    "Maharashtra": "South & SE Asia", "Orissa": "South & SE Asia", "Odisha": "South & SE Asia",
    "Chola": "South & SE Asia",
    # Cities
    "Delhi": "South & SE Asia", "Mumbai": "South & SE Asia", "Bombay": "South & SE Asia",
    "Calcutta": "South & SE Asia", "Kolkata": "South & SE Asia", "Madras": "South & SE Asia",
    "Chennai": "South & SE Asia", "Bangalore": "South & SE Asia", "Hyderabad": "South & SE Asia",
    "Bangkok": "South & SE Asia", "Hanoi": "South & SE Asia", "Saigon": "South & SE Asia",
    "Jakarta": "South & SE Asia", "Manila": "South & SE Asia", "Phnom Penh": "South & SE Asia",
    "Yogyakarta": "South & SE Asia", "Angkor": "South & SE Asia", "Kathmandu": "South & SE Asia",

    # ============ MIDDLE EAST & N. AFRICA ============
    "Iran": "Middle East & N. Africa", "Iranian": "Middle East & N. Africa",
    "Persia": "Middle East & N. Africa", "Persian": "Middle East & N. Africa",
    "Iraq": "Middle East & N. Africa", "Iraqi": "Middle East & N. Africa",
    "Mesopotamia": "Middle East & N. Africa", "Mesopotamian": "Middle East & N. Africa",
    "Sumer": "Middle East & N. Africa", "Sumerian": "Middle East & N. Africa",
    "Assyria": "Middle East & N. Africa", "Assyrian": "Middle East & N. Africa",
    "Babylon": "Middle East & N. Africa", "Babylonian": "Middle East & N. Africa",
    "Akkad": "Middle East & N. Africa", "Akkadian": "Middle East & N. Africa",
    "Sasanian": "Middle East & N. Africa", "Achaemenid": "Middle East & N. Africa",
    "Parthian": "Middle East & N. Africa", "Elamite": "Middle East & N. Africa",
    "Turkey": "Middle East & N. Africa", "Turkish": "Middle East & N. Africa",
    "Anatolia": "Middle East & N. Africa", "Anatolian": "Middle East & N. Africa",
    "Ottoman": "Middle East & N. Africa", "Hittite": "Middle East & N. Africa",
    "Syria": "Middle East & N. Africa", "Syrian": "Middle East & N. Africa",
    "Lebanon": "Middle East & N. Africa", "Lebanese": "Middle East & N. Africa",
    "Phoenicia": "Middle East & N. Africa", "Phoenician": "Middle East & N. Africa",
    "Israel": "Middle East & N. Africa", "Israeli": "Middle East & N. Africa",
    "Palestine": "Middle East & N. Africa", "Palestinian": "Middle East & N. Africa",
    "Jordan": "Middle East & N. Africa", "Jordanian": "Middle East & N. Africa",
    "Egypt": "Middle East & N. Africa", "Egyptian": "Middle East & N. Africa",
    "Coptic": "Middle East & N. Africa",
    "Morocco": "Middle East & N. Africa", "Moroccan": "Middle East & N. Africa",
    "Tunisia": "Middle East & N. Africa", "Tunisian": "Middle East & N. Africa",
    "Algeria": "Middle East & N. Africa", "Algerian": "Middle East & N. Africa",
    "Libya": "Middle East & N. Africa", "Libyan": "Middle East & N. Africa",
    "Saudi Arabia": "Middle East & N. Africa", "Saudi": "Middle East & N. Africa",
    "Yemen": "Middle East & N. Africa", "Yemeni": "Middle East & N. Africa",
    "Oman": "Middle East & N. Africa", "Kuwait": "Middle East & N. Africa",
    "Bahrain": "Middle East & N. Africa", "Qatar": "Middle East & N. Africa",
    "United Arab Emirates": "Middle East & N. Africa", "UAE": "Middle East & N. Africa",
    "Sudan": "Middle East & N. Africa", "Sudanese": "Middle East & N. Africa",
    "Nubia": "Middle East & N. Africa", "Nubian": "Middle East & N. Africa",
    "Mauritania": "Middle East & N. Africa",
    "Islamic": "Middle East & N. Africa",
    "Arab": "Middle East & N. Africa", "Arabic": "Middle East & N. Africa",
    "Arabia": "Middle East & N. Africa",
    # Cities
    "Cairo": "Middle East & N. Africa", "Alexandria": "Middle East & N. Africa",
    "Luxor": "Middle East & N. Africa",
    "Baghdad": "Middle East & N. Africa", "Mosul": "Middle East & N. Africa",
    "Nineveh": "Middle East & N. Africa", "Ur": "Middle East & N. Africa",
    "Tehran": "Middle East & N. Africa", "Isfahan": "Middle East & N. Africa",
    "Shiraz": "Middle East & N. Africa", "Tabriz": "Middle East & N. Africa",
    "Istanbul": "Middle East & N. Africa", "Constantinople": "Middle East & N. Africa",
    "Damascus": "Middle East & N. Africa", "Aleppo": "Middle East & N. Africa",
    "Beirut": "Middle East & N. Africa", "Tyre": "Middle East & N. Africa",
    "Sidon": "Middle East & N. Africa",
    "Jerusalem": "Middle East & N. Africa", "Tel Aviv": "Middle East & N. Africa",
    "Petra": "Middle East & N. Africa",
    "Mecca": "Middle East & N. Africa", "Medina": "Middle East & N. Africa",
    "Marrakech": "Middle East & N. Africa", "Fez": "Middle East & N. Africa",
    "Casablanca": "Middle East & N. Africa", "Tangier": "Middle East & N. Africa",
    "Tunis": "Middle East & N. Africa", "Carthage": "Middle East & N. Africa",
    "Algiers": "Middle East & N. Africa",

    # ============ SUB-SAHARAN AFRICA ============
    "Africa": "Sub-Saharan Africa", "African": "Sub-Saharan Africa",
    "Nigeria": "Sub-Saharan Africa", "Nigerian": "Sub-Saharan Africa",
    "Yoruba": "Sub-Saharan Africa", "Igbo": "Sub-Saharan Africa", "Hausa": "Sub-Saharan Africa",
    "Benin": "Sub-Saharan Africa",
    "Kenya": "Sub-Saharan Africa", "Kenyan": "Sub-Saharan Africa",
    "Ghana": "Sub-Saharan Africa", "Ghanaian": "Sub-Saharan Africa", "Ashanti": "Sub-Saharan Africa",
    "Mali": "Sub-Saharan Africa", "Malian": "Sub-Saharan Africa", "Dogon": "Sub-Saharan Africa",
    "Bambara": "Sub-Saharan Africa",
    "Senegal": "Sub-Saharan Africa", "Senegalese": "Sub-Saharan Africa",
    "Ethiopia": "Sub-Saharan Africa", "Ethiopian": "Sub-Saharan Africa",
    "South Africa": "Sub-Saharan Africa", "South African": "Sub-Saharan Africa",
    "Cameroon": "Sub-Saharan Africa", "Cameroonian": "Sub-Saharan Africa",
    "Congo": "Sub-Saharan Africa", "Congolese": "Sub-Saharan Africa",
    "Democratic Republic of the Congo": "Sub-Saharan Africa",
    "Tanzania": "Sub-Saharan Africa", "Tanzanian": "Sub-Saharan Africa",
    "Uganda": "Sub-Saharan Africa", "Ugandan": "Sub-Saharan Africa",
    "Zimbabwe": "Sub-Saharan Africa", "Zimbabwean": "Sub-Saharan Africa",
    "Zambia": "Sub-Saharan Africa", "Mozambique": "Sub-Saharan Africa",
    "Angola": "Sub-Saharan Africa", "Botswana": "Sub-Saharan Africa", "Namibia": "Sub-Saharan Africa",
    "Madagascar": "Sub-Saharan Africa", "Rwanda": "Sub-Saharan Africa", "Burundi": "Sub-Saharan Africa",
    "Niger": "Sub-Saharan Africa", "Chad": "Sub-Saharan Africa",
    "Liberia": "Sub-Saharan Africa", "Sierra Leone": "Sub-Saharan Africa",
    "Burkina Faso": "Sub-Saharan Africa", "Ivory Coast": "Sub-Saharan Africa",
    "Cote d'Ivoire": "Sub-Saharan Africa", "Côte d'Ivoire": "Sub-Saharan Africa",
    "Togo": "Sub-Saharan Africa", "Gabon": "Sub-Saharan Africa", "Eritrea": "Sub-Saharan Africa",
    "Somalia": "Sub-Saharan Africa", "Malawi": "Sub-Saharan Africa",
    "Lesotho": "Sub-Saharan Africa", "Eswatini": "Sub-Saharan Africa",
    "Guinea": "Sub-Saharan Africa", "Cape Verde": "Sub-Saharan Africa",
    "Gambia": "Sub-Saharan Africa", "Central African Republic": "Sub-Saharan Africa",

    # ============ OCEANIA ============
    "Australia": "Oceania", "Australian": "Oceania", "Aboriginal": "Oceania",
    "New Zealand": "Oceania", "Maori": "Oceania",
    "Papua New Guinea": "Oceania", "Papuan": "Oceania", "Melanesia": "Oceania",
    "Polynesia": "Oceania", "Polynesian": "Oceania", "Micronesia": "Oceania",
    "Fiji": "Oceania", "Fijian": "Oceania", "Samoa": "Oceania", "Samoan": "Oceania",
    "Tonga": "Oceania", "Tongan": "Oceania", "Tahiti": "Oceania", "Tahitian": "Oceania",
    "Hawaii": "Oceania", "Hawaiian": "Oceania",
    "New Caledonia": "Oceania", "Solomon Islands": "Oceania", "Vanuatu": "Oceania",
    "Sydney": "Oceania", "Melbourne": "Oceania", "Auckland": "Oceania", "Wellington": "Oceania",

    # ============ MANUALLY ADDED REGION MAPPING ============
    "Unknown Place": "Unknown", "Europe": "Europe", "Staffordshire": "Europe","Burslem": "Europe",
    "Saint-Louis-lès-Bitche": "Europe",
    "Clichy": "Europe", "Evanston": "North America","Sèvres": "Europe", "Meissen": "Europe",
    "Mediterranean Region": "Mediterranean","Birmingham": "Europe", "Baccarat": "Europe",
    "Lunéville": "Europe","Guna Yala": "Latin America & Caribbean",
    "Jouy-en-Josas": "Europe","Corning": "North America", "Tulsa": "North America", "Venezia province": "Europe",
    "Highland Park": "North America", "Trier": "Europe",
    "Southeast Asia": "South & SE Asia", "Lake Forest": "North America",
    "Eastern Mediterranean Region": "Mediterranean", "Nantes": "Europe", "Midwest": "North America", "Ancient Mediterranean": "Mediterranean",
    "Schleswig": "Europe", "Thessalía": "Europe","Rouen": "Europe","Winnetka": "North America",
    "Navajo": "North America", "Puebla": "Latin America & Caribbean",
  
}

# Pre-sort keys by length descending so longer/more specific names match first.
# This is what fixes collisions like "Indiana" vs "India" or "South America" vs "America".
_sorted_keys = sorted(country_to_region.keys(), key=len, reverse=True)

def assign_region(place):
    if pd.isna(place):
        return "Unknown"
    if place in country_to_region:
        return country_to_region[place]
        
    # Substring fallback for entries like "Paris, France" or "Chicago, Illinois"
    place_str = str(place)
    for key in _sorted_keys:
        if key in place_str:
            return country_to_region[key]
    return "Unmapped"

art["region"] = art["place_of_origin"].apply(assign_region)

# era buckets
def to_era(year):
    if pd.isna(year): return "Unknown"
    if year < -1000:  return "Ancient: pre-1000 BCE"
    if year < 0:      return "Classical: 1000 BCE-0"
    if year < 500:    return "Late Ancient: 0-500"
    if year < 1000:   return "Early Medieval: 500-1000"
    if year < 1400:   return "Late Medieval: 1000-1400"
    if year < 1600:   return "Renaissance: 1400-1600"
    if year < 1800:   return "Early Modern: 1600-1800"
    if year < 1850:   return "Early 19th c.: 1800-1850"
    if year < 1900:   return "Late 19th c.: 1850-1900"
    if year < 1945:   return "Early 20th c.: 1900-1945"
    if year < 1970:   return "Postwar: 1945-1970"
    if year < 2000:   return "Late 20th c.: 1970-2000"
    return "Contemporary: 2000+"
art["era"] = art["year_start"].apply(to_era)

# binary exhibition flag
art["has_exhibition_history"] = (
    art["exhibition_history"].notna()
    & (art["exhibition_history"].astype(str).str.strip().str.len() > 5)).astype(int)

# rename department for brevity to match the queries below
art = art.rename(columns={"department_title": "department"})

# accession year recovered from the reference number ("1880.1" -> 1880); regex-free + dtype-safe
def _acc_year(ref):
    if pd.isna(ref): return np.nan
    head = str(ref).strip()[:4]
    return int(head) if head.isdigit() and 1600 <= int(head) <= 2026 else np.nan
art["accession_year"] = art["main_reference_number"].apply(_acc_year)

# ---- Second dataset: world population by region (approx 2024, millions) ----
region_population_m = {
    "East Asia": 1640, "South & SE Asia": 2600, "Europe": 745,
    "Sub-Saharan Africa": 1200, "Middle East & N. Africa": 530,
    "North America": 505, "Latin America & Caribbean": 535, "Oceania": 45,
}
population = (pd.Series(region_population_m, name="population_m")
              .rename_axis("region").reset_index())
population["pop_share_pct"] = (population["population_m"]
                               / population["population_m"].sum() * 100).round(2)

# ---- Load into SQLite (flatten any list columns first) ----
con = sqlite3.connect(":memory:")
art_sql = art.copy()
for c in art_sql.columns:
    if art_sql[c].apply(lambda v: isinstance(v, list)).any():
        art_sql[c] = art_sql[c].apply(lambda v: ", ".join(map(str, v)) if isinstance(v, list) else v)
art_sql.to_sql("artworks", con, index=False, if_exists="replace")
population.to_sql("region_population", con, index=False, if_exists="replace")

# ---- Third dataset: the museum's FULL public catalogue (real, ~134k works) from GitHub ----
import json, urllib.request
real_path = Path("aic_real"); real_path.mkdir(exist_ok=True)
jf = real_path / "allArtworks.jsonl"
if not jf.exists():
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/art-institute-of-chicago/api-data/"
        "master/getting-started/allArtworks.jsonl", jf)
catalogue = pd.DataFrame(json.loads(l) for l in open(jf, encoding="utf-8"))
catalogue["accession_year"] = catalogue["main_reference_number"].apply(_acc_year)
catalogue = catalogue.rename(columns={"department_title": "department",
                                      "artist_title": "artist"})
catalogue.to_sql("catalogue", con, index=False, if_exists="replace")

def run(sql):
    """Helper: execute a SQL string and return the result as a DataFrame."""
    return pd.read_sql_query(sql, con)

print(f"Loaded {len(art_sql):,} cache artworks, {len(catalogue):,} full-catalogue works, "
      f"and {len(population)} population rows into SQLite.")

Loaded 131,564 cache artworks, 134,078 full-catalogue works, and 8 population rows into SQLite.


### Data-validity sanity check (GROUP BY / aggregates)

In [2]:
run("""
-- WHAT: Row count, distinct regions, distinct departments, and overall exhibition rate.
-- HOW : Plain aggregate functions over the whole table (no grouping key).
-- WHY : A first validity pass before any analysis - confirms the table loaded, the
--        derived columns are populated, and the exhibition flag is a sane 0..1 average.
-- OUTPUT: a single summary row.
SELECT COUNT(*)                          AS n_rows,
       COUNT(DISTINCT region)            AS n_regions,
       COUNT(DISTINCT department)        AS n_departments,
       ROUND(AVG(has_exhibition_history)*100, 1) AS pct_exhibited
FROM artworks;
""")

,n_rows,n_regions,n_departments,pct_exhibited
0,131564,11,15,10.1


### Q2 — Works and exhibition rate by region (GROUP BY #1)

In [3]:
run("""
-- WHAT: For each region, the number of works and the share with an exhibition history.
-- HOW : GROUP BY region with COUNT and AVG.
-- WHY : The core composition cut - shows the Europe/North-America concentration and how
--        visibility (exhibition rate) differs across regions.
-- OUTPUT: one row per region, largest first.
SELECT region,
       COUNT(*)                                   AS n_works,
       ROUND(AVG(has_exhibition_history)*100, 1)  AS pct_exhibited
FROM artworks
GROUP BY region
ORDER BY n_works DESC;
""")

,region,n_works,pct_exhibited
0,Europe,51755,10.0
1,North America,45967,9.5
2,East Asia,18583,5.0
3,Unknown,4671,11.0
4,Unmapped,3457,18.1
5,Middle East & N. Africa,2427,20.5
6,Latin America & Caribbean,2232,21.1
7,South & SE Asia,1276,19.7
8,Sub-Saharan Africa,955,36.4
9,Mediterranean,162,80.9


### Q3 — Composition by region and era (GROUP BY #2)

In [4]:
run("""
-- WHAT: Counts for every region x era combination.
-- HOW : GROUP BY on two columns, with HAVING to drop empty combos.
-- WHY : Surfaces the historical 'mix' of each region - e.g. whether a region's holdings
--        cluster in one or two eras (a research/attribution signal for the committee).
-- OUTPUT: one row per non-empty region/era pair.
SELECT region, era, COUNT(*) AS n
FROM artworks
GROUP BY region, era
HAVING COUNT(*) > 0
ORDER BY region, n DESC;
""")

,region,era,n
0,East Asia,Early Modern: 1600-1800,5018
1,East Asia,Unknown,2878
2,East Asia,Early 19th c.: 1800-1850,2244
3,East Asia,Late 19th c.: 1850-1900,2042
4,East Asia,Postwar: 1945-1970,1606
...,...,...,...
131,Unmapped,Unknown,79
132,Unmapped,Late Ancient: 0-500,75
133,Unmapped,Contemporary: 2000+,71
134,Unmapped,Early Medieval: 500-1000,57


### Q4 — Rank departments by size (WINDOW #1: RANK)

In [5]:
run("""
-- WHAT: Departments ordered by holdings, with an explicit rank.
-- HOW : A subquery aggregates counts; RANK() OVER (ORDER BY ...) assigns the ranking.
-- WHY : Gives the committee a clean 'league table' of departments by collection size.
-- OUTPUT: one row per department with size_rank = 1 for the largest.
SELECT department,
       n_works,
       RANK() OVER (ORDER BY n_works DESC) AS size_rank
FROM (SELECT department, COUNT(*) AS n_works FROM artworks GROUP BY department)
ORDER BY size_rank;
""")

,department,n_works,size_rank
0,Prints and Drawings,53267,1
1,Photography and Media,24567,2
2,Arts of Asia,16460,3
3,Textiles,11771,4
4,Architecture and Design,6176,5
5,Applied Arts of Europe,5606,6
6,Arts of the Americas,4386,7
7,Painting and Sculpture of Europe,2372,8
8,"Arts of Greece, Rome, and Byzantium",2196,9
9,Contemporary Art,1859,10


### Q5 — Each region's share of the collection (WINDOW #2: SUM OVER)

In [6]:
run("""
-- WHAT: Region counts expressed as a percentage of the mapped-region total.
-- HOW : SUM(n_works) OVER () computes the grand total in the same pass, so each row can
--        be divided by it without a second query or a manual constant.
-- WHY : Turns raw counts into shares - the unit the representation analysis needs.
-- OUTPUT: one row per region with pct_of_collection summing to ~100.
SELECT region,
       n_works,
       ROUND(100.0 * n_works / SUM(n_works) OVER (), 2) AS pct_of_collection
FROM (SELECT region, COUNT(*) AS n_works
      FROM artworks
      WHERE region NOT IN ('Unknown','Unmapped','Mediterranean')
      GROUP BY region)
ORDER BY n_works DESC;
""")

,region,n_works,pct_of_collection
0,Europe,51755,41.98
1,North America,45967,37.29
2,East Asia,18583,15.07
3,Middle East & N. Africa,2427,1.97
4,Latin America & Caribbean,2232,1.81
5,South & SE Asia,1276,1.04
6,Sub-Saharan Africa,955,0.77
7,Oceania,79,0.06


### Q6 — Dominant era within each region (WINDOW #3: ROW_NUMBER + PARTITION)

In [7]:
run("""
-- WHAT: Ranks eras inside each region from most to least common.
-- HOW : ROW_NUMBER() OVER (PARTITION BY region ORDER BY n DESC) restarts the count for
--        every region, so era_rank_in_region = 1 is that region's single biggest era.
-- WHY : Lets the committee read, at a glance, what each region's collection is 'made of'
--        historically - and where 'Unknown' ranks high, flagging attribution gaps.
-- OUTPUT: region/era rows with a per-region rank; filter to rank=1 for the headline era.
SELECT region, era, n,
       ROW_NUMBER() OVER (PARTITION BY region ORDER BY n DESC) AS era_rank_in_region
FROM (SELECT region, era, COUNT(*) AS n FROM artworks GROUP BY region, era)
ORDER BY region, era_rank_in_region;
""")

,region,era,n,era_rank_in_region
0,East Asia,Early Modern: 1600-1800,5018,1
1,East Asia,Unknown,2878,2
2,East Asia,Early 19th c.: 1800-1850,2244,3
3,East Asia,Late 19th c.: 1850-1900,2042,4
4,East Asia,Postwar: 1945-1970,1606,5
...,...,...,...,...
131,Unmapped,Unknown,79,10
132,Unmapped,Late Ancient: 0-500,75,11
133,Unmapped,Contemporary: 2000+,71,12
134,Unmapped,Early Medieval: 500-1000,57,13


### Q7 — Collection vs. population by region (JOIN #1)

In [8]:
run("""
-- WHAT: Brings the external population table alongside the collection counts.
-- HOW : JOIN artworks to region_population on region; GROUP BY to aggregate works.
-- WHY : This is the join that makes the representation question answerable - it puts
--        'how many works' next to 'how many people' for each region.
-- OUTPUT: one row per region with works, population, and population share.
SELECT p.region,
       COUNT(a.id)            AS n_works,
       p.population_m,
       ROUND(p.pop_share_pct, 2) AS pop_share_pct
FROM artworks a
JOIN region_population p ON a.region = p.region
GROUP BY p.region, p.population_m, p.pop_share_pct
ORDER BY n_works DESC;
""")

,region,n_works,population_m,pop_share_pct
0,Europe,51755,745,9.55
1,North America,45967,505,6.47
2,East Asia,18583,1640,21.03
3,Middle East & N. Africa,2427,530,6.79
4,Latin America & Caribbean,2232,535,6.86
5,South & SE Asia,1276,2600,33.33
6,Sub-Saharan Africa,955,1200,15.38
7,Oceania,79,45,0.58


### Q8 — Representation index (JOIN #2: join to a derived CTE + window)

In [9]:
run("""
-- WHAT: The representation index = collection share / population share, in pure SQL.
-- HOW : A CTE computes each region's collection share with SUM() OVER (); that derived
--        table is JOINed back to region_population to divide the two shares.
-- WHY : Reproduces the headline metric from the Python notebook entirely in SQL - a
--        cross-check that the index isn't an artifact of the pandas code path.
-- OUTPUT: one row per region; index > 1 = over-represented, < 1 = under-represented.
WITH coll AS (
    SELECT region,
           COUNT(*) AS n,
           100.0 * COUNT(*) / SUM(COUNT(*)) OVER () AS coll_share
    FROM artworks
    WHERE region IN (SELECT region FROM region_population)
    GROUP BY region)
SELECT c.region,
       c.n                                     AS n_works,
       ROUND(c.coll_share, 2)                  AS coll_share_pct,
       ROUND(p.pop_share_pct, 2)               AS pop_share_pct,
       ROUND(c.coll_share / p.pop_share_pct, 2) AS representation_index
FROM coll c
JOIN region_population p ON c.region = p.region
ORDER BY representation_index DESC;
""")

,region,n_works,coll_share_pct,pop_share_pct,representation_index
0,North America,45967,37.29,6.47,5.76
1,Europe,51755,41.98,9.55,4.40
2,East Asia,18583,15.07,21.03,0.72
3,Middle East & N. Africa,2427,1.97,6.79,0.29
4,Latin America & Caribbean,2232,1.81,6.86,0.26
5,Oceania,79,0.06,0.58,0.11
6,Sub-Saharan Africa,955,0.77,15.38,0.05
7,South & SE Asia,1276,1.04,33.33,0.03


### Q9 — Recent acquisitions vs. population (JOIN #3)

In [10]:
run("""
-- WHAT: Last-decade acquisitions (FY>=2016) and total holdings for each region, next to
--        that region's population.
-- HOW : JOIN to region_population; conditional aggregation (SUM of a CASE) counts recent
--        accessions in the same pass as the total.
-- WHY : Tests whether recent acquisition behaviour is flowing toward the populous,
--        under-represented regions or simply reinforcing the existing concentration.
-- OUTPUT: one row per region with acquired_last10, total, and population.
SELECT p.region,
       p.population_m,
       SUM(CASE WHEN a.fiscal_year >= 2016 THEN 1 ELSE 0 END) AS acquired_last10,
       COUNT(*)                                               AS total_holdings
FROM artworks a
JOIN region_population p ON a.region = p.region
GROUP BY p.region, p.population_m
ORDER BY acquired_last10 DESC;
""")

,region,population_m,acquired_last10,total_holdings
0,North America,505,4969,45967
1,Europe,745,3669,51755
2,East Asia,1640,3009,18583
3,Sub-Saharan Africa,1200,385,955
4,Latin America & Caribbean,535,344,2232
5,South & SE Asia,2600,301,1276
6,Middle East & N. Africa,530,101,2427
7,Oceania,45,5,79


### Q10 — Above-average regions (SUBQUERY #1: non-correlated IN)

In [11]:
run("""
-- WHAT: Keeps only regions whose holdings exceed the average region size.
-- HOW : A non-correlated subquery in the IN clause returns the list of 'big' regions;
--        the inner HAVING compares each region's count to a scalar subquery (avg size).
-- WHY : A quick way to separate the handful of dominant regions from the long tail.
-- OUTPUT: the regions larger than the mean region, with their counts.
SELECT region, COUNT(*) AS n_works
FROM artworks
WHERE region IN (
        SELECT region FROM artworks
        GROUP BY region
        HAVING COUNT(*) > (SELECT COUNT(*) * 1.0 / COUNT(DISTINCT region) FROM artworks))
GROUP BY region
ORDER BY n_works DESC;
""")

,region,n_works
0,Europe,51755
1,North America,45967
2,East Asia,18583


### Q11 — Works older than their department average (SUBQUERY #2: correlated)

In [12]:
run("""
-- WHAT: For each department, how many works predate that department's own average year.
-- HOW : A CORRELATED scalar subquery recomputes AVG(year_start) for the matching
--        department (b.department = a.department) for every outer row.
-- WHY : A relative-age view - shows which departments hold a large 'older than typical'
--        tail, useful when thinking about conservation or historical depth.
-- OUTPUT: one row per department with the count of below-average-year works.
-- NOTE: the per-department averages are pre-aggregated in a CTE first, so the correlated
--        subquery scans ~10 rows per outer row instead of the whole 131k-row table - same
--        result, but it returns instantly on the full catalogue.
WITH dept_avg AS (
    SELECT department, AVG(year_start) AS avg_year
    FROM artworks WHERE year_start IS NOT NULL
    GROUP BY department)
SELECT a.department, COUNT(*) AS n_older_than_dept_avg
FROM artworks a
WHERE a.year_start < (SELECT d.avg_year FROM dept_avg d WHERE d.department = a.department)
GROUP BY a.department
ORDER BY n_older_than_dept_avg DESC;
""")

,department,n_older_than_dept_avg
0,Prints and Drawings,18994
1,Photography and Media,8985
2,Textiles,3491
3,Architecture and Design,2905
4,Arts of Asia,2437
5,Applied Arts of Europe,2299
6,Arts of the Americas,1194
7,"Arts of Greece, Rome, and Byzantium",1027
8,Arts of Africa,945
9,Contemporary Art,899


### Q12 — Cumulative acquisitions over time (SUBQUERY in FROM + WINDOW)

In [13]:
run("""
-- WHAT: Annual accession counts since 1990 plus a running cumulative total.
-- HOW : A subquery in the FROM clause produces yearly counts; SUM(...) OVER (ORDER BY
--        fiscal_year) accumulates them into a running total.
-- WHY : Shows the pace of growth and where big bequests created step-changes - context
--        for how much volume a future acquisition cycle should target.
-- OUTPUT: one row per fiscal year with n_acq and cumulative_acq.
SELECT fiscal_year,
       n_acq,
       SUM(n_acq) OVER (ORDER BY fiscal_year) AS cumulative_acq
FROM (SELECT CAST(fiscal_year AS INT) AS fiscal_year, COUNT(*) AS n_acq
      FROM artworks
      WHERE fiscal_year IS NOT NULL AND fiscal_year >= 1990
      GROUP BY CAST(fiscal_year AS INT))
ORDER BY fiscal_year;
""")

,fiscal_year,n_acq,cumulative_acq
0,1990,2325,2325
1,1991,1592,3917
2,1992,1510,5427
3,1993,650,6077
4,1994,548,6625
5,1995,841,7466
6,1996,738,8204
7,1997,983,9187
8,1998,943,10130
9,1999,1157,11287


### Q13 — Department size quartiles (WINDOW: NTILE)

In [14]:
run("""
-- WHAT: Rank departments into four size tiers (quartiles).
-- HOW : NTILE(4) OVER (ORDER BY n_works DESC) buckets the grouped counts.
-- WHY : A compact way to talk about 'top-tier vs bottom-tier' departments without
--        hard-coding thresholds - useful for prioritising where to grow holdings.
-- OUTPUT: one row per department with its size and quartile (1 = largest).
SELECT department,
       n_works,
       NTILE(4) OVER (ORDER BY n_works DESC) AS size_quartile
FROM (SELECT department, COUNT(*) AS n_works
      FROM artworks WHERE department IS NOT NULL
      GROUP BY department)
ORDER BY n_works DESC;
""")

,department,n_works,size_quartile
0,Prints and Drawings,53267,1
1,Photography and Media,24567,1
2,Arts of Asia,16460,1
3,Textiles,11771,1
4,Architecture and Design,6176,2
5,Applied Arts of Europe,5606,2
6,Arts of the Americas,4386,2
7,Painting and Sculpture of Europe,2372,2
8,"Arts of Greece, Rome, and Byzantium",2196,3
9,Contemporary Art,1859,3


### Q14 — Real acquisition timeline with a moving total (WINDOW FRAME on the full catalogue)

In [15]:
run("""
-- WHAT: Accessions per decade across the FULL 134k catalogue, plus a 3-decade moving sum.
-- HOW : Aggregate the real `catalogue` table by decade, then a windowed SUM with an explicit
--        frame: ROWS BETWEEN 2 PRECEDING AND CURRENT ROW.
-- WHY : The full catalogue (not just the cache) is the honest basis for sizing growth; the
--        moving total smooths one-off bequest spikes to show the underlying trend.
-- OUTPUT: one row per decade with that decade's count and the trailing 3-decade total.
SELECT decade,
       n_acq,
       SUM(n_acq) OVER (ORDER BY decade ROWS BETWEEN 2 PRECEDING AND CURRENT ROW) AS trailing_3dec
FROM (SELECT (CAST(accession_year AS INT)/10)*10 AS decade, COUNT(*) AS n_acq
      FROM catalogue
      WHERE accession_year IS NOT NULL AND accession_year >= 1880
      GROUP BY (CAST(accession_year AS INT)/10)*10)
ORDER BY decade;
""")

,decade,n_acq,trailing_3dec
0,1880,425,425
1,1890,1213,1638
2,1900,1121,2759
3,1910,2944,5278
4,1920,18100,22165
5,1930,7388,28432
6,1940,8547,34035
7,1950,8628,24563
8,1960,10629,27804
9,1970,10881,30138


### Q15 — Representation tiers (JOIN + CASE segmentation)

In [16]:
run("""
-- WHAT: Label each region Over- / Roughly- / Under-represented vs world population.
-- HOW : JOIN the per-region collection share to the population table, then a CASE expression
--        segments the representation ratio into three committee-friendly tiers.
-- WHY : Turns the continuous representation index into the plain-language buckets the
--        acquisition committee can act on directly.
-- OUTPUT: one row per region with its index and tier.
WITH region_share AS (
    SELECT region,
           COUNT(*) AS n_works,
           100.0 * COUNT(*) / (SELECT COUNT(*) FROM artworks WHERE region IS NOT NULL) AS coll_share_pct
    FROM artworks WHERE region IS NOT NULL
    GROUP BY region)
SELECT r.region,
       r.n_works,
       ROUND(r.coll_share_pct, 2)                       AS coll_share_pct,
       p.pop_share_pct,
       ROUND(r.coll_share_pct / p.pop_share_pct, 2)      AS representation_index,
       CASE WHEN r.coll_share_pct / p.pop_share_pct >= 1.5 THEN 'Over-represented'
            WHEN r.coll_share_pct / p.pop_share_pct >= 0.67 THEN 'Roughly proportional'
            ELSE 'Under-represented' END                 AS tier
FROM region_share r
JOIN region_population p ON r.region = p.region
ORDER BY representation_index DESC;
""")

,region,n_works,coll_share_pct,pop_share_pct,representation_index,tier
0,North America,45967,34.94,6.47,5.40,Over-represented
1,Europe,51755,39.34,9.55,4.12,Over-represented
2,East Asia,18583,14.12,21.03,0.67,Roughly proportional
3,Middle East & N. Africa,2427,1.84,6.79,0.27,Under-represented
4,Latin America & Caribbean,2232,1.70,6.86,0.25,Under-represented
5,Oceania,79,0.06,0.58,0.10,Under-represented
6,Sub-Saharan Africa,955,0.73,15.38,0.05,Under-represented
7,South & SE Asia,1276,0.97,33.33,0.03,Under-represented


---
### Summary

These fifteen queries reproduce the report's headline numbers directly in SQL and add a few
relational views (department ranking, within-region era ranking, cumulative accessions) that
are awkward in pandas but natural in SQL. The representation index in **Q8** matches the value
computed in the Python notebook, and the region counts in **Q2/Q5** match the EDA charts —
the cross-engine agreement is the main data-quality guarantee here.